In [1]:
#!C:\PYTHON_ENV\env_object_detection\Scripts\pip.exe uninstall torch torchvision torchaudio -y

In [2]:
data_dir = r"C:\Tutorial\Visuara\Computer vision from Scratch\flowers"

In [3]:
# !pip uninstall -y torch torchvision torchaudio
# !pip install torch torchvision torchaudio

In [4]:
import sys
print(sys.executable)

C:\PYTHON_ENV\env_object_detection\Scripts\python.exe


In [5]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\PYTHON_ENV\env_object_detection\Scripts\python.exe -m pip install --upgrade pip


In [6]:
# !pip install torchsummary

# !pip install torchinfo

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchvision import models

from torchsummary import summary

import wandb
import os

In [8]:
import os
import torch
from torchvision.models import resnet50, ResNet50_Weights

# FIX: Bypass the corrupted cache by setting a new temporary download folder
os.environ['TORCH_HOME'] = './clean_pytorch_models'

print("Downloading fresh model weights...")
# Load the pre-trained model
model = resnet50(weights=ResNet50_Weights.DEFAULT)
print("Success! ResNet50 downloaded and ready.")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to ./clean_pytorch_models\hub\checkpoints\resnet50-11ad3fa6.pth


100%|█████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:34<00:00, 2.95MB/s]


Success! ResNet50 downloaded and ready.


In [9]:
# =======================
# STEP 0: Initialize wandb
# =======================
wandb.init(project="ResNet-flowers-v2", config={
    "epochs": 20,
    "batch_size": 16,
    "learning_rate": 0.001,
    "architecture": "ResNet",
    "pretrained": True,
    "input_size": 128
})

# Shortcut to config values
config = wandb.config

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\sangr\_netrc.
wandb: Currently logged in as: sangram01 (sangram01-srm-institute-of-science-and-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [10]:
# =======================
# STEP 1: Data Preparation
# =======================

# Transforms for training and validation
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

train_dir = r"C:\Tutorial\Visuara\Computer vision from Scratch\flowers/train"
val_dir = r"C:\Tutorial\Visuara\Computer vision from Scratch\flowers/val"

train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms['train'])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size)

In [11]:
import torch.nn as nn

# 1. Freeze all the pre-trained feature layers FIRST
for param in model.parameters():
    param.requires_grad = False

# 2. Replace the final classifier layer for your 5 flower classes
# (New layers have requires_grad=True by default, so it's ready to train)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5)

# 3. Move the model to the correct device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = model.to(device)

# Watch the model's weights and gradients in Weights & Biases
wandb.watch(model, log="all", log_freq=10)

Using device: cpu


In [12]:
# ===================
# STEP 3: Loss & Optimizer
# ===================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

In [13]:
def train_model(model, criterion, optimizer, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_correct = 0
        train_total = 0
        running_loss = 0.0

        print(f"\nEpoch {epoch + 1}/{epochs}")
        print("-" * 30)

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            batch_correct = (preds == labels).sum().item()
            train_correct += batch_correct
            train_total += labels.size(0)

            # Print every 10 batches
            if (i + 1) % 10 == 0:
                batch_acc = batch_correct / labels.size(0)
                print(f"[Batch {i+1}/{len(train_loader)}] Loss: {loss.item():.4f}, Batch Acc: {batch_acc:.4f}")

        train_acc = train_correct / train_total
        wandb.log({"epoch": epoch + 1, "train_loss": running_loss, "train_accuracy": train_acc})
        print(f"Epoch {epoch+1} Summary - Loss: {running_loss:.4f}, Train Accuracy: {train_acc:.4f}")

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        wandb.log({"epoch": epoch + 1, "val_accuracy": val_acc})
        print(f"Validation Accuracy: {val_acc:.4f}")


In [14]:
# ===================
# Train the model
# ===================
train_model(model, criterion, optimizer, train_loader, 
            val_loader, epochs=config.epochs)


Epoch 1/20
------------------------------
[Batch 10/251] Loss: 1.4912, Batch Acc: 0.3750
[Batch 20/251] Loss: 1.2983, Batch Acc: 0.6250
[Batch 30/251] Loss: 1.2597, Batch Acc: 0.6875
[Batch 40/251] Loss: 0.8299, Batch Acc: 0.9375
[Batch 50/251] Loss: 1.0646, Batch Acc: 0.7500
[Batch 60/251] Loss: 0.9351, Batch Acc: 0.7500
[Batch 70/251] Loss: 0.7824, Batch Acc: 0.8750


KeyboardInterrupt: 

In [15]:
import torch
print(torch.version.cuda)

None


In [16]:
import sys
!{sys.executable} -m pip uninstall torch torchvision torchaudio -y
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Found existing installation: torch 2.11.0

You can safely remove it manually.

Uninstalling torch-2.11.0:


  Successfully uninstalled torch-2.11.0

You can safely remove it manually.

...


  File "C:\PYTHON_ENV\env_object_detection\Lib\site-packages\pip\_vendor\urllib3\response.py", line 443, in _error_catcher


   ---------------------------------------- 0.0/2.8 GB 1.2 MB/s eta 0:38:07


    raise ReadTimeoutError(self._pool, None, "Read timed out.")


pip._vendor.urllib3.exceptions.ReadTimeoutError: HTTPSConnectionPool(host='download.pytorch.org', port=443): Read timed out.


   ---------------------------------------- 0.0/2.8 GB 1.2 MB/s eta 0:38:19
   ---------------------------------------- 0.0/2.8 GB 1.2 MB/s eta 0:38:19
   ---------------------------------------- 0.0/2.8 GB 1.2 MB/s eta 0:38:19
   ---------------------------------------- 0.0/2.8 GB 1.2 MB/s eta 0:38:19
   ---------------------------------------- 0.0/2.8 GB 1.2 MB/s eta 0:40:16
   ---------------------------------------- 0.0/2.8 GB 1.2 MB/s eta 0:40:07
   ---------------------------------------- 0.0/2.8 GB 1.2 MB/s eta 0:40:07
   ---------------------------------------- 0.0/2.8 GB 1.2 MB/s eta 0:40:07
   ---------------------------------------- 0.0/2.8 GB 1.2 MB/s eta 0:40:07
   ---------------------------------------- 0.0/2.8 GB 1.1 MB/s eta 0:42:03
   ---------------------------------------- 0.0/2.8 GB 1.1 MB/s eta 0:42:03
   ---------------------------------------- 0.0/2.8 GB 1.1 MB/s eta 0:42:03
   ---------------------------------------- 0.0/2.8 GB 1.1 MB/s eta 0:42:03
   ---------

In [17]:
torch.cuda.is_available()

False

In [ ]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --default-timeout=1000

Looking in indexes: https://download.pytorch.org/whl/cu118


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\PYTHON_ENV\env_object_detection\Scripts\python.exe -m pip install --upgrade pip

  Using cached https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata (27 kB)

ERROR: THESE PACKAGES DO NOT MATCH THE HASHES FROM THE REQUIREMENTS FILE. If you have updated the package versions, please update the hashes. Otherwise, examine the package contents carefully; someone may have tampered with them.

    unknown package:


  Using cached https://download-r2.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata (6.3 kB)


        Expected sha256 80855ec840b7b06372ff43535d01393a8ec101842618d1f9ed629572b52aed71

             Got        834fbe5552d0bdcf565065d5b191e254d47d91dd0562082d662ac84b8a77d3aa

  Using cached https://download-r2.pytorch.org/whl/cu118/torchaudio-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata (6.8 kB)



   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB 266.4 kB/s eta 2:56:13
   ---------------------------------------- 0.0/2.8 GB 266.4 kB/s eta 2:56:13
   ---------------------------------------- 0.0/2.8 GB 266.4 kB/s eta 2:56:13
   ---------------------------------------- 0.0/2.8 GB 266.4 kB/s eta 2:56:13
   ---------------------------------------- 0.0/2.8 GB 279.6 kB/s 